# 🧠 Adam Optimizer: การรวมเทคนิค Momentum และ RMSProp เข้าด้วยกัน

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Adam Optimizer**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายคณิตศาสตร์ของ Adam: การติดตามโมเมนต์แรก (momentum) และโมเมนต์ที่สอง (RMSProp) รวมถึงการปรับแก้ค่าอคติ (bias correction)
2. อิมพลีเมนต์ **Adam Optimizer จากศูนย์ (from scratch)** ใน Python/NumPy
3. เปรียบเทียบเส้นทางการลู่เข้าของ **Vanilla GD**, **Momentum GD** และ **Adam** บนฟังก์ชันต้นทุนแบบหุบเหว 2 มิติที่ลาดชัน:
   $$f(x, y) = 0.5x^2 + 10y^2$$
4. แสดงภาพเส้นทางการลู่เข้าบนแผนที่เส้นชั้นความสูงแบบ 2 มิติ เพื่อสังเกตวิธีที่ Adam ลู่เข้าสู่คำตอบได้อย่างรวดเร็ว ราบรื่น และมีการปรับเปลี่ยนอัตราการเรียนรู้ให้เหมาะกับแต่ละพิกัด
5. พล็อตเส้นกราฟการลดลงของลอสเพื่อเปรียบเทียบความเร็วในการลู่เข้า
6. อธิบายเกี่ยวกับ **AdamW (Decoupled Weight Decay)** และบทบาทของมันในโครงข่ายประสาทเทียมเชิงลึกสมัยใหม่อย่าง YOLO

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลย

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. การกำหนดฟังก์ชันหุบเหว (Ravine Function) และเกรเดียนต์

ฟังก์ชันต้นทุนของเราแสดงถึงหุบเขาลึก:
$$f(x, y) = 0.5x^2 + 10y^2$$

เกรเดียนต์:
$$\frac{\partial f}{\partial x} = x, \quad \frac{\partial f}{\partial y} = 20y$$

In [ ]:
def cost_ravine(x, y):
    return 0.5 * x**2 + 10.0 * y**2

def grad_ravine(x, y):
    return np.array([x, 20.0 * y])

## 2. การอิมพลีเมนต์ตัวปรับค่า (Optimizers) จากศูนย์

ลองเขียนลูปการเพิ่มประสิทธิภาพทั้งสามแบบ:
1.  **Vanilla GD:** $\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \nabla J(\mathbf{w}_t)$
2.  **Momentum:** ใช้เวกเตอร์ความเร็วสะสมเพื่อเร่งความเร็วไปตามก้นหุบเขา
3.  **Adam:** ผสมผสาน momentum ($m_t$) และค่าเฉลี่ยเกรเดียนต์ยกกำลังสอง ($v_t$) โดยประยุกต์ใช้การปรับแก้ค่าอคติในช่วงเริ่มต้น (time-step power bias corrections)

In [ ]:
def optimize_vanilla(start_pos, lr=0.15, epochs=50):
    pos = np.array(start_pos, dtype=float)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        pos -= lr * grad
        history.append(pos.copy())
    return np.array(history)

def optimize_momentum(start_pos, lr=0.15, beta=0.9, epochs=50):
    pos = np.array(start_pos, dtype=float)
    v = np.zeros(2)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        v = beta * v + lr * grad
        pos -= v
        history.append(pos.copy())
    return np.array(history)

def optimize_adam(start_pos, lr=0.15, beta1=0.9, beta2=0.999, eps=1e-8, epochs=50):
    pos = np.array(start_pos, dtype=float)
    m = np.zeros(2)
    v = np.zeros(2)
    history = [pos.copy()]
    
    for t in range(1, epochs + 1):
        grad = grad_ravine(pos[0], pos[1])
        
        # 1. Update first moment
        m = beta1 * m + (1.0 - beta1) * grad
        
        # 2. Update second moment
        v = beta2 * v + (1.0 - beta2) * (grad ** 2)
        
        # 3. Bias correction
        m_hat = m / (1.0 - beta1 ** t)
        v_hat = v / (1.0 - beta2 ** t)
        
        # 4. Parameter update
        pos -= (lr / (np.sqrt(v_hat) + eps)) * m_hat
        history.append(pos.copy())
        
    return np.array(history)

# Run optimizations starting at (8.0, 4.0)
start = [8.0, 4.0]
path_vanilla = optimize_vanilla(start, lr=0.08)
path_momentum = optimize_momentum(start, lr=0.08, beta=0.8)
path_adam = optimize_adam(start, lr=0.25)

## 3. การแสดงภาพเส้นทางการลู่เข้าบนแผนที่เส้นชั้นความสูง (Contour Map)

มาสร้างกริดเส้นชั้นความสูงแบบ 2 มิติและพล็อตเส้นทางการลู่เข้ากัน

In [ ]:
x = np.linspace(-10, 10, 150)
y = np.linspace(-5, 5, 150)
X, Y = np.meshgrid(x, y)
Z = cost_ravine(X, Y)

plt.figure(figsize=(12, 8))
contours = plt.contour(X, Y, Z, levels=30, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

# Plot paths
plt.plot(path_vanilla[:, 0], path_vanilla[:, 1], color='red', marker='o', alpha=0.8, linewidth=1.5, label='Vanilla GD')
plt.plot(path_momentum[:, 0], path_momentum[:, 1], color='blue', marker='s', alpha=0.8, linewidth=1.5, label='Momentum GD')
plt.plot(path_adam[:, 0], path_adam[:, 1], color='green', marker='x', linewidth=2.5, label='Adam (Rapid, Smooth)')

plt.scatter(0, 0, color='gold', s=150, marker='*', zorder=5, label='Minimum (0,0)')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(-10, 10)
plt.ylim(-5, 5)
plt.title('Comparison of Optimizers: Vanilla vs. Momentum vs. Adam')
plt.legend()
plt.show()

ดูพล็อตสิ!
-   **Vanilla GD (สีแดง):** แกว่งไปมาอย่างรุนแรงเนื่องจากผนังที่สูงชันในแนวแกน y
-   **Momentum (สีน้ำเงิน):** ช่วยปรับให้การแกว่งในแนวตั้งราบรื่นขึ้น และเร่งความเร็วขึ้นไปตามพื้นราบในแนวนอน
-   **Adam (สีเขียว):** ลู่เข้าหาเป้าหมายได้รวดเร็วและตรงประเด็นยิ่งขึ้นไปอีก! การปรับเปลี่ยนสัดส่วนอัตราการเรียนรู้ของแต่ละพิกัดช่วยลดขนาดก้าวเดินในแนวตั้ง (แกน $y$) ลงในทันที ในขณะที่ก้าวเดินในแนวนอน (แกน $x$) ยังคงกว้างอยู่ ส่งผลให้เคลื่อนที่เป็นเส้นทแยงมุมเข้าสู่ศูนย์กลางได้อย่างสมบูรณ์แบบ

## 4. การเปรียบเทียบความเร็วในการลู่เข้า

ลองพล็อตเส้นกราฟการลดลงของต้นทุนกัน

In [ ]:
cost_vanilla = [cost_ravine(p[0], p[1]) for p in path_vanilla]
cost_momentum = [cost_ravine(p[0], p[1]) for p in path_momentum]
cost_adam = [cost_ravine(p[0], p[1]) for p in path_adam]

plt.figure(figsize=(10, 5))
plt.plot(cost_vanilla, color='red', label='Vanilla GD')
plt.plot(cost_momentum, color='blue', label='Momentum GD')
plt.plot(cost_adam, color='green', label='Adam')
plt.yscale('log')
plt.xlabel('Steps')
plt.ylabel('Log Cost')
plt.title('Cost Convergence Comparison (Log Scale)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 💡 ความเชื่อมโยงกับ YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **AdamW (Decoupled Weight Decay):** ในระหว่างการฝึก โมเดล YOLO จะใช้งาน **AdamW** แทนที่จะเป็น Adam แบบปกติทั่วไป ใน Adam แบบปกตินั้น เกรเดียนต์การทำโทษของ L2 weight decay ($\lambda \mathbf{w}$) จะถูกรวมเข้าโดยตรงกับค่าเฉลี่ยโมเมนต์แรกและโมเมนต์ที่สอง ($m_t$ และ $v_t$) ซึ่งจะทำให้การอัปเดตน้ำหนักจาก weight decay ผิดเพี้ยนไป ส่วน AdamW จะแก้ปัญหานี้โดยการหักลบ weight decay เข้ากับพารามิเตอร์น้ำหนักโดยตรง:
    $$\mathbf{w}_{t+1} = \mathbf{w}_t - \text{Adam\_update} - \alpha \lambda \mathbf{w}_t$$
    วิธีนี้ช่วยรักษาสัดส่วนทางคณิตศาสตร์ที่ถูกต้องของระบบควบคุมน้ำหนักส่วนเกิน (weight regularization) ส่งผลให้การลู่เข้ามีเสถียรภาพมากกว่ามาก